# 任务 6：模型量化与 ONNX 部署准备

本 Notebook 加载任务 5 的 ResNet50 人脸特征模型，完成 PyTorch 动态量化、模型大小与 CPU 延迟对比、LFW 精度复测，以及 ONNX 导出和 ONNX Runtime 推理验证。

## 6.1 优化方法

- **量化**：用更低位宽表示权重或激活，减少模型大小并改善特定硬件上的推理速度。本实验按任务要求使用 PyTorch 动态量化。
- **剪枝**：移除重要性较低的连接、通道或结构；非结构化稀疏只有在硬件和推理引擎支持时才会带来实际加速。
- **知识蒸馏**：让较小的学生模型学习教师模型的输出或中间特征，以更少计算量保留精度。

动态量化会把 `Linear` 权重保存为 int8，并在推理时动态量化激活。ResNet50 的主要计算来自卷积，因此收益可能有限，但它能以最小改动建立可复现的精度、速度和大小基线。

In [1]:
from io import BytesIO
from pathlib import Path
from time import perf_counter
import json
import sys

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader

current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in (current_dir, *current_dir.parents) if (path / 'face_recognition').exists())
sys.path.insert(0, str(PROJECT_ROOT / 'face_recognition'))

from arcface import (
    FaceEmbeddingModel, ImagePathDataset, evaluate_lfw,
    evaluation_transform, extract_embeddings, read_lfw_pairs,
)

CONFIG = json.loads((PROJECT_ROOT / 'face_recognition' / 'arcface_config.json').read_text(encoding='utf-8'))
WORK_DIR = PROJECT_ROOT / 'work_dirs' / 'arcface_resnet50'
EXPORT_DIR = PROJECT_ROOT / 'output' / 'models'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = WORK_DIR / 'latest.pth'
assert CHECKPOINT.exists(), '请先运行 07_arcface_training.ipynb'

In [2]:
float_model = FaceEmbeddingModel(embedding_dim=CONFIG['embedding_dim'], pretrained=False).cpu().eval()
checkpoint = torch.load(CHECKPOINT, map_location='cpu')
float_model.load_state_dict(checkpoint['model'])
example = torch.randn(1, 3, 112, 112)
with torch.inference_mode():
    assert float_model(example).shape == (1, CONFIG['embedding_dim'])
print('已加载 epoch:', checkpoint['epoch'])

已加载 epoch: 10


## 6.2 PyTorch 动态量化

In [3]:
quantized_model = torch.ao.quantization.quantize_dynamic(
    float_model, {nn.Linear}, dtype=torch.qint8,
).eval()

with torch.inference_mode():
    float_output = float_model(example)
    quantized_output = quantized_model(example)
print('输出最大绝对误差:', (float_output - quantized_output).abs().max().item())

输出最大绝对误差: 0.012127263471484184


In [4]:
def state_dict_size_mb(model):
    stream = BytesIO()
    torch.save(model.state_dict(), stream)
    return stream.tell() / 1024**2

def benchmark_torch(model, batch, warmup=20, repeats=100):
    model.eval()
    with torch.inference_mode():
        for _ in range(warmup):
            model(batch)
        started = perf_counter()
        for _ in range(repeats):
            model(batch)
    return (perf_counter() - started) * 1000 / repeats

benchmark_batch = torch.randn(1, 3, 112, 112)
comparison = {
    'float': {
        'size_mb': state_dict_size_mb(float_model),
        'latency_ms': benchmark_torch(float_model, benchmark_batch),
    },
    'dynamic_int8': {
        'size_mb': state_dict_size_mb(quantized_model),
        'latency_ms': benchmark_torch(quantized_model, benchmark_batch),
    },
}
comparison

{'float': {'size_mb': 93.98734474182129, 'latency_ms': 16.108950999987428},
 'dynamic_int8': {'size_mb': 90.98813438415527,
  'latency_ms': 17.187645000012708}}

### LFW 精度对比

使用与任务 5 完全相同的 10 折协议，确认量化没有造成不可接受的精度下降。

In [5]:
LFW_ROOT = PROJECT_ROOT / 'data' / 'lfw' / 'lfw-deepfunneled'
PAIRS_CSV = PROJECT_ROOT / 'data' / 'lfw' / 'pairs.csv'
pairs = read_lfw_pairs(PAIRS_CSV, LFW_ROOT)
unique_paths = sorted({path for pair in pairs for path in pair[:2]})
lfw_loader = DataLoader(
    ImagePathDataset(unique_paths, evaluation_transform()),
    batch_size=128, num_workers=4,
)
for name, model in [('float', float_model), ('dynamic_int8', quantized_model)]:
    embeddings = extract_embeddings(model, lfw_loader, torch.device('cpu'))
    result = evaluate_lfw(pairs, embeddings)
    comparison[name]['lfw_accuracy'] = result['accuracy']
    comparison[name]['lfw_std'] = result['std']
    print(name, f"{result['accuracy']:.4%} ± {result['std']:.4%}")

float 80.1833% ± 2.3111%
dynamic_int8 80.2167% ± 2.3083%


## 6.3 导出 ONNX 并验证推理

导出浮点特征模型，输入批量维度设为动态；随后用 ONNX Runtime 对同一输入推理，并与 PyTorch 输出比较。

In [6]:
ONNX_PATH = EXPORT_DIR / 'arcface_resnet50.onnx'
torch.onnx.export(
    float_model, example, ONNX_PATH,
    input_names=['images'], output_names=['embeddings'],
    dynamic_axes={'images': {0: 'batch'}, 'embeddings': {0: 'batch'}},
    opset_version=17, do_constant_folding=True,
)
print(ONNX_PATH, f'{ONNX_PATH.stat().st_size / 1024**2:.2f} MB')

D:\InternProject\face-vision-lab\output\models\arcface_resnet50.onnx 93.61 MB


In [7]:
import onnx
import onnxruntime as ort

onnx.checker.check_model(onnx.load(ONNX_PATH))
session = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
onnx_output = session.run(None, {'images': example.numpy()})[0]
onnx_error = float(np.max(np.abs(onnx_output - float_output.numpy())))
print('ONNX 输出最大绝对误差:', onnx_error)
assert onnx_error < 1e-4

ONNX 输出最大绝对误差: 1.8998980522155762e-07


In [8]:
def benchmark_onnx(session, batch, warmup=20, repeats=100):
    values = {'images': batch.numpy()}
    for _ in range(warmup):
        session.run(None, values)
    started = perf_counter()
    for _ in range(repeats):
        session.run(None, values)
    return (perf_counter() - started) * 1000 / repeats

comparison['onnx'] = {
    'size_mb': ONNX_PATH.stat().st_size / 1024**2,
    'latency_ms': benchmark_onnx(session, benchmark_batch),
    'max_abs_error': onnx_error,
}
torch.save(quantized_model.state_dict(), EXPORT_DIR / 'arcface_resnet50_dynamic_int8.pth')
(WORK_DIR / 'optimization_results.json').write_text(
    json.dumps(comparison, indent=2), encoding='utf-8',
)
comparison

{'float': {'size_mb': 93.98734474182129,
  'latency_ms': 16.108950999987428,
  'lfw_accuracy': 0.8018333333333333,
  'lfw_std': 0.023111444976220967},
 'dynamic_int8': {'size_mb': 90.98813438415527,
  'latency_ms': 17.187645000012708,
  'lfw_accuracy': 0.8021666666666667,
  'lfw_std': 0.023082581215183803},
 'onnx': {'size_mb': 93.61047458648682,
  'latency_ms': 4.892196000000695,
  'max_abs_error': 1.8998980522155762e-07}}

## 实验结论

| 模型 | 大小 | CPU 单图延迟 | LFW Accuracy |
|---|---:|---:|---:|
| PyTorch FP32 | 93.99 MB | 16.11 ms | 80.1833% ± 2.3111% |
| PyTorch 动态 INT8 | 90.99 MB | 17.19 ms | 80.2167% ± 2.3083% |
| ONNX Runtime FP32 | 93.61 MB | 4.89 ms | 未复测 |

- 动态量化后模型缩小 3.00 MB（3.19%），特征最大绝对误差为 0.0121；LFW Accuracy 变化仅 +0.033 个百分点，属于验证波动，可视为精度基本保持。
- 动态量化的 CPU 延迟反而增加 6.70%。原因是本实验只动态量化末端 Linear 层，而 ResNet50 的主要计算量来自卷积层，因此模型大小和速度收益都有限，不能把该结果表述为整个网络的 INT8 量化。
- ONNX 模型通过结构检查和 ONNX Runtime 推理验证，与 PyTorch 输出的最大绝对误差仅为 1.90×10⁻⁷。其 CPU 单图延迟为 4.89 ms，相对 PyTorch FP32 降低 69.63%，约为 3.29 倍加速，因此 ONNX Runtime 是本实验中更有效的部署方案。完整对比结果保存在 `optimization_results.json`，导出模型位于 `output/models/arcface_resnet50.onnx`。